# Otpor glatke kugle: predvidi → izračunaj → provjeri

Dimenzijska analiza daje oblik \(C_D=f(Re)\), ali ne daje samu funkciju niti jamči da je jedna korelacija valjana u svim režimima. Ovdje koristimo glatku aproksimaciju za neporemećen tok oko glatke kugle samo u rasponu \(0{,}1\le Re\le2\cdot10^5\), prije područja krize otpora.

## Predvidi

1. Zašto iz jednakog Reynoldsova broja slijedi jednak \(C_D\) samo ako su ostali uvjeti sličnosti usporedivi?
2. Hoće li linearna interpolacija po \(Re\) i log–log interpolacija dati isti rezultat između rijetkih tabličnih točaka?
3. Što treba učiniti kada je traženi \(Re\) izvan kalibriranog raspona: ekstrapolirati bez upozorenja ili zaustaviti račun?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RE_MIN, RE_MAX = 0.1, 2.0e5

def cd_correlation(Re):
    # Didaktička aproksimacija glatke kugle unutar eksplicitnog raspona.
    Re = np.asarray(Re, dtype=float)
    if np.any((Re < RE_MIN) | (Re > RE_MAX)):
        raise ValueError(f"Model nije dopušten izvan {RE_MIN:g} ≤ Re ≤ {RE_MAX:g}.")
    return 24/Re*(1+0.15*Re**0.687) + 0.42/(1+4.25e4*Re**(-1.16))

def loglog_interpolate(Re, re_table, cd_table):
    Re = np.asarray(Re, dtype=float)
    if np.any((Re < re_table[0]) | (Re > re_table[-1])):
        raise ValueError("Interpolacija nije ekstrapolacija: Re je izvan tablice.")
    return np.exp(np.interp(np.log(Re), np.log(re_table), np.log(cd_table)))

re_nodes = np.geomspace(RE_MIN, RE_MAX, 14)
cd_nodes = cd_correlation(re_nodes)
re_dense = np.geomspace(RE_MIN, RE_MAX, 800)
cd_true = cd_correlation(re_dense)
cd_log = loglog_interpolate(re_dense, re_nodes, cd_nodes)
cd_linear = np.interp(re_dense, re_nodes, cd_nodes)

relative_log_error = np.abs(cd_log/cd_true-1)
relative_method_difference = np.abs(cd_linear/cd_log-1)
print(f"Najveća pogreška rijetke log–log tablice: {100*relative_log_error.max():.2f} %")
print(f"Najveća razlika linearne i log–log interpolacije: {100*relative_method_difference.max():.2f} %")


## Izračunaj: radna točka i interpolacijska osjetljivost

Za zrak računamo \(Re\), interpolirani \(C_D\) i silu. Zatim tablicu prorjeđujemo i zgušnjavamo. Promjena rezultata je procjena **interpolacijske**, a ne eksperimentalne ni modelske nesigurnosti.


In [ ]:
rho, nu, velocity, diameter = 1.20, 1.50e-5, 0.60, 0.030
Re_work = velocity*diameter/nu
Cd_work = float(loglog_interpolate(Re_work, re_nodes, cd_nodes))
area = np.pi*diameter**2/4
drag = 0.5*rho*velocity**2*Cd_work*area

node_counts = np.array([8, 12, 18, 30, 50])
cd_by_resolution = []
for count in node_counts:
    re_tab = np.geomspace(RE_MIN, RE_MAX, int(count))
    cd_tab = cd_correlation(re_tab)
    cd_by_resolution.append(float(loglog_interpolate(Re_work, re_tab, cd_tab)))
cd_by_resolution = np.asarray(cd_by_resolution)

print(f"Radna točka: Re={Re_work:.0f}, Cd={Cd_work:.4f}, F_D={1e3*drag:.4f} mN")
print("Cd pri broju tabličnih točaka:", dict(zip(node_counts, np.round(cd_by_resolution, 5))))


## Provjeri

Provjeravamo reprodukciju poznatih tabličnih čvorova, približavanje Stokesovu graničnom zakonu pri malom \(Re\) i obvezno zaustavljanje izvan raspona. Posljednja provjera sprječava prividno preciznu ekstrapolaciju kroz krizu otpora, gdje hrapavost i turbulencija slobodnog toka postaju bitne.


In [ ]:
assert np.allclose(loglog_interpolate(re_nodes, re_nodes, cd_nodes), cd_nodes, rtol=1e-13)
assert abs(float(cd_correlation(RE_MIN))/(24/RE_MIN)-1) < 0.04
outside_blocked = False
try:
    cd_correlation(5e5)
except ValueError:
    outside_blocked = True
assert outside_blocked
assert drag > 0 and RE_MIN <= Re_work <= RE_MAX

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(re_dense, cd_true, color="#256d85", lw=2, label="korelacija")
axes[0].loglog(re_nodes, cd_nodes, "o", color="#b43c35", label="rijetka tablica")
axes[0].loglog(re_dense, cd_log, "--", color="#d28f2c", label="log–log interpolacija")
axes[0].set(xlabel="Re", ylabel="$C_D$", title="Model samo u označenom rasponu")
axes[0].legend(fontsize=8)
axes[1].semilogx(re_dense, 100*relative_method_difference, color="#7a3e9d")
axes[1].set(xlabel="Re", ylabel="razlika metoda (%)", title="Osjetljivost na interpolaciju")
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Veći broj tabličnih točaka smanjuje samo numeričku interpolacijsku pogrešku. Ne uklanja pogrešku korelacije niti nadomješta mjerenja za hrapavu kuglu, blizinu stijenke ili turbulentni slobodni tok.
